In [1]:
import json
import uuid
from sqlalchemy import create_engine

from utils import reset_db, get_session, model_to_dict
from data.models import udahub

# Udahub Application

## Core Database

**Init DB**

In [2]:
udahub_db = "data/core/udahub.db"

In [3]:
reset_db(udahub_db)

2026-08-16 20:47:46,490 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-08-16 20:47:46,491 INFO sqlalchemy.engine.Engine COMMIT
✅ Recreated data/core/udahub.db with fresh schema


In [4]:
engine = create_engine(f"sqlite:///{udahub_db}", echo=False)
udahub.Base.metadata.create_all(bind=engine)

**Account**

In [5]:
account_id = "cultpass"
account_name = "CultPass Card"

In [6]:
with get_session(engine) as session:
    account = udahub.Account(
        account_id=account_id,
        account_name=account_name,
    )
    session.add(account)

## Integrations

**Knowledge Base**

In [7]:
# TODO: Create additional 10 articles


In [7]:
cultpass_articles = []

with open('data/external/cultpass_articles.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        cultpass_articles.append(json.loads(line))

In [ ]:
cultpass_articles

In [9]:
if len(cultpass_articles) < 14:
    raise AssertionError("You should load the articles with at least 14 records")

In [10]:
with get_session(engine) as session:
    kb = []
    for article in cultpass_articles:
        knowledge = udahub.Knowledge(
            article_id=str(uuid.uuid4()),
            account_id=account_id,
            title=article["title"],
            content=article["content"],
            tags=article["tags"]
        )
        kb.append(knowledge)
    session.add_all(kb) 
    

**Ticket**

In [11]:
cultpass_users = []

with open('data/external/cultpass_users.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        cultpass_users.append(json.loads(line))

In [12]:
ticket_info = {
    "status": "open",
    "content": "I can't log in to my Cultpass account.",
    "owner_id": cultpass_users[0]["id"],
    "owner_name": cultpass_users[0]["name"],
    "role": "user",
    "channel": "chat",
    "tags": "login, access",
}

In [13]:
with get_session(engine) as session:
    user = session.query(udahub.User).filter_by(
        account_id=account_id,
        external_user_id=ticket_info["owner_id"],
    ).first()

    if not user:
        user = udahub.User(
            user_id=str(uuid.uuid4()),
            account_id=account_id,
            external_user_id=ticket_info["owner_id"],
            user_name=ticket_info["owner_name"],
        )
    
    ticket = udahub.Ticket(
        ticket_id=str(uuid.uuid4()),
        account_id=account_id,
        user_id=user.user_id,
        channel=ticket_info["channel"],
    )
    metadata = udahub.TicketMetadata(
        ticket_id=ticket.ticket_id,
        status=ticket_info["status"],
        main_issue_type=None,
        tags=ticket_info["tags"],
    )

    first_message = udahub.TicketMessage(
        message_id=str(uuid.uuid4()),
        ticket_id=ticket.ticket_id,
        role=ticket_info["role"],
        content=ticket_info["content"],
    )

    session.add_all([user, ticket, metadata, first_message])


# Tests

In [14]:
with get_session(engine) as session:
    account = session.query(udahub.Account).filter_by(
        account_id=account_id
    ).first()
    print(account)

<Account(account_id='cultpass', account_name='CultPass Card')>


In [15]:
with get_session(engine) as session:
    account = session.query(udahub.Account).filter_by(
        account_id=account_id
    ).first()
    for article in account.knowledge_articles:
        print(article)

<Knowledge(article_id='354cd6f2-fb8a-4213-b1e8-1b2a7642eda9', title='How to Reserve a Spot for an Event')>
<Knowledge(article_id='c65f6a2d-64d0-4ab9-a0d6-a39792de367d', title='What's Included in a CultPass Subscription')>
<Knowledge(article_id='7749b7f4-dd3d-484a-9629-47d202359832', title='How to Cancel or Pause a Subscription')>
<Knowledge(article_id='9e582317-5e41-4d6a-acbe-ba06cbdb7237', title='How to Handle Login Issues?')>
<Knowledge(article_id='5f365a91-b837-4c5f-89a4-f4f3512742f8', title='How to View Upcoming Reservations')>
<Knowledge(article_id='2a1a4c1c-303e-4e26-bd2f-770fe53617b9', title='What to Do If a Reservation Is Full')>
<Knowledge(article_id='12a6a834-9bf2-4437-9a50-73e9002537dc', title='How to Cancel an Event Reservation')>
<Knowledge(article_id='ad3c17ae-bbf2-48f7-8959-d3ad366c190c', title='What Happens If I Miss an Event')>
<Knowledge(article_id='58e2a2a6-8565-4c0b-a22c-042c750c65ed', title='How to Access My QR Code')>
<Knowledge(article_id='af6da06c-a1b4-4add-9da4

In [16]:
with get_session(engine) as session:
    users = session.query(udahub.User).all()
    for user in users:
        print(user)

<User(user_id='9220ee17-7e78-4915-80c1-857dafab578e', user_name='Alice Kingsley', external_user_id='a4ab87')>


In [17]:
with get_session(engine) as session:
    user = session.query(udahub.User).filter_by(
        account_id=account_id,
        external_user_id=ticket_info["owner_id"],
    ).first()
    
    ticket:udahub.Ticket = user.tickets[0]
    for message in ticket.messages:
        print(message)

<TicketMessage(message_id='7cdd7cef-8fab-42f0-b2ed-69687e10e8cd', role='user', content='I can't log in to my Cultpass ...')>
